# Notebook 01 — EDA Out-of-Core e Qualidade de Dados (*The Complete Journey*)

**Objetivo:** Explorar os 8 arquivos CSV oficiais da base *dunnhumby — The Complete Journey* usando exclusivamente `pl.scan_csv()` (*LazyFrame* + *streaming* no Polars), comprovando:
1. **Volumetria e Assimetria (~82% em `causal_data.csv`)** — por que `causal_data.csv` (695,86 MB) vai para o Apache Cassandra.
2. **Cobertura Demográfica Polimórfica** — 2.500 domicílios transacionam em `transaction_data.csv`, mas apenas 801 possuem registro em `hh_demographic.csv` (`classification_1..5`).
3. **Códigos Alfanuméricos Promocionais** — `display` (`'0'..'9'`, `'A'`) e `mailer` (`'0'`, `'A'`, `'D'`, `'Z'`, etc.) exigem tipo CQL `text`.
4. **Fórmulas Oficiais de Preço (PDF pág. 3)** — diferença entre `SALES_VALUE`, *Loyalty card price* e *Non-loyalty price*.
5. **Estudo de Caso Oficial (`household_key = 208`, John Smith, PDF págs. 2 e 7–9)** — linha do tempo completa de campanhas, compras e resgate do cupom `51800000050` na Campanha 22 (Dia 654).

### Fluxo Out-of-Core de Leitura (Zero Full-Dataset In-Memory)
```text
[8 CSVs dunnhumby (~808 MiB)] ──► get_settings().csv_path(fname)
                                    └──► pl.scan_csv(schema_overrides=TRANSACTION/CAUSAL_SCHEMA_OVERRIDES)
                                           └──► .collect(engine="streaming") ──► Destino Poliglota:
                                                  ├─► [MongoDB]   households (2.500 perfis, 801 c/ demographics)
                                                  ├─► [Cassandra] 4 tabelas CQL (tx, tx_product, sales, causal)
                                                  └─► [Redis]     campanhas/cupons ativos c/ TTL + rankings
```

In [1]:
import polars as pl
from basketlens.config import get_settings
from basketlens.ingestion.cassandra_loader import CAUSAL_SCHEMA_OVERRIDES
from basketlens.ingestion.mongo_loader import TRANSACTION_SCHEMA_OVERRIDES

settings = get_settings()
print(f"Dataset directory: {settings.resolved_dataset_dir}")

Dataset directory: /home/dmattos/PersonalDevelop/basketlens/dataset


## 1. Armadilha 1: Volumetria dos 8 CSVs e Assimetria do `causal_data.csv` (~82%)

In [2]:
csv_files = [
    "transaction_data.csv",
    "causal_data.csv",
    "product.csv",
    "coupon.csv",
    "campaign_table.csv",
    "coupon_redempt.csv",
    "hh_demographic.csv",
    "campaign_desc.csv",
]

rows_meta = []
total_bytes = sum(settings.csv_path(f).stat().st_size for f in csv_files)
for fname in csv_files:
    fpath = settings.csv_path(fname)
    size_mb = fpath.stat().st_size / (1024 * 1024)
    pct = (fpath.stat().st_size / total_bytes) * 100.0
    overrides = (
        TRANSACTION_SCHEMA_OVERRIDES
        if fname == "transaction_data.csv"
        else CAUSAL_SCHEMA_OVERRIDES
        if fname == "causal_data.csv"
        else {"COUPON_UPC": pl.Utf8}
        if fname in ("coupon.csv", "coupon_redempt.csv")
        else None
    )
    lf = pl.scan_csv(fpath, schema_overrides=overrides)
    row_count = lf.select(pl.len()).collect(engine="streaming").item()
    col_count = len(lf.collect_schema())
    rows_meta.append(
        {
            "file": fname,
            "rows": row_count,
            "cols": col_count,
            "size_mb": round(size_mb, 3),
            "volume_pct": round(pct, 2),
        }
    )

volume_df = pl.DataFrame(rows_meta)
print(volume_df)

shape: (8, 5)
┌──────────────────────┬──────────┬──────┬─────────┬────────────┐
│ file                 ┆ rows     ┆ cols ┆ size_mb ┆ volume_pct │
│ ---                  ┆ ---      ┆ ---  ┆ ---     ┆ ---        │
│ str                  ┆ i64      ┆ i64  ┆ f64     ┆ f64        │
╞══════════════════════╪══════════╪══════╪═════════╪════════════╡
│ transaction_data.csv ┆ 2595732  ┆ 12   ┆ 135.176 ┆ 16.73      │
│ causal_data.csv      ┆ 36786524 ┆ 5    ┆ 663.622 ┆ 82.15      │
│ product.csv          ┆ 92353    ┆ 7    ┆ 6.132   ┆ 0.76       │
│ coupon.csv           ┆ 124548   ┆ 3    ┆ 2.692   ┆ 0.33       │
│ campaign_table.csv   ┆ 7208     ┆ 3    ┆ 0.091   ┆ 0.01       │
│ coupon_redempt.csv   ┆ 2318     ┆ 4    ┆ 0.052   ┆ 0.01       │
│ hh_demographic.csv   ┆ 801      ┆ 8    ┆ 0.041   ┆ 0.01       │
│ campaign_desc.csv    ┆ 30       ┆ 4    ┆ 0.001   ┆ 0.0        │
└──────────────────────┴──────────┴──────┴─────────┴────────────┘


## 2. Armadilha 2: Cobertura Demográfica Incompleta (801 de 2.500 Domicílios)

Justifica o polimorfismo de documentos na coleção `households` do MongoDB: armazenamos todos os **2.500** domicílios com `behavior_summary` e `recent_baskets`, mas apenas **801** incluem o subdocumento `demographics`.

In [3]:
tx_lf = pl.scan_csv(
    settings.csv_path("transaction_data.csv"),
    schema_overrides=TRANSACTION_SCHEMA_OVERRIDES,
)
demo_lf = pl.scan_csv(settings.csv_path("hh_demographic.csv"))

total_hh_tx = tx_lf.select(pl.col("household_key").n_unique()).collect(engine="streaming").item()
total_hh_demo = demo_lf.select(pl.col("household_key").n_unique()).collect(engine="streaming").item()

print({
    "distinct_households_in_transactions": total_hh_tx,
    "distinct_households_with_demographics": total_hh_demo,
    "households_without_demographics": total_hh_tx - total_hh_demo,
    "demographic_coverage_pct": round((total_hh_demo / total_hh_tx) * 100, 2),
})

print("\nDistribuição por Faixa Etária (classification_1) e Nível de Renda (classification_3):")
print(
    demo_lf.group_by("classification_1")
    .len()
    .sort("classification_1")
    .collect()
)

{'distinct_households_in_transactions': 2500, 'distinct_households_with_demographics': 801, 'households_without_demographics': 1699, 'demographic_coverage_pct': 32.04}

Distribuição por Faixa Etária (classification_1) e Nível de Renda (classification_3):
shape: (6, 2)
┌──────────────────┬─────┐
│ classification_1 ┆ len │
│ ---              ┆ --- │
│ str              ┆ u32 │
╞══════════════════╪═════╡
│ Age Group1       ┆ 46  │
│ Age Group2       ┆ 142 │
│ Age Group3       ┆ 194 │
│ Age Group4       ┆ 288 │
│ Age Group5       ┆ 59  │
│ Age Group6       ┆ 72  │
└──────────────────┴─────┘


## 3. Armadilha 3: Códigos Alfanuméricos em `causal_data.csv` (`display` e `mailer` como `text`)

In [4]:
causal_lf = pl.scan_csv(
    settings.csv_path("causal_data.csv"),
    schema_overrides=CAUSAL_SCHEMA_OVERRIDES,
)
display_codes = (
    causal_lf.select(pl.col("display").unique().sort())
    .collect(engine="streaming")["display"]
    .to_list()
)
mailer_codes = (
    causal_lf.select(pl.col("mailer").unique().sort())
    .collect(engine="streaming")["mailer"]
    .to_list()
)
print("Unique display codes (alphanumeric):", display_codes)
print("Unique mailer codes (alphanumeric):", mailer_codes)

Unique display codes (alphanumeric): ['0', '1', '2', '3', '4', '5', '6', '7', '9', 'A']
Unique mailer codes (alphanumeric): ['0', 'A', 'C', 'D', 'F', 'H', 'J', 'L', 'P', 'X', 'Z']


## 4. Fórmulas Oficiais de Preço (PDF pág. 3: *Loyalty Card Price* vs. *Non-Loyalty Price*)

- `Loyalty card price = (SALES_VALUE - (RETAIL_DISC + COUPON_MATCH_DISC)) / QUANTITY`
- `Non-loyalty price = (SALES_VALUE - COUPON_MATCH_DISC) / QUANTITY`

In [5]:
price_examples_df = (
    tx_lf.filter(
        (pl.col("QUANTITY") > 0)
        & (pl.col("RETAIL_DISC") < 0)
        & (pl.col("COUPON_MATCH_DISC") < 0)
    )
    .with_columns(
        (
            (pl.col("SALES_VALUE") - (pl.col("RETAIL_DISC") + pl.col("COUPON_MATCH_DISC")))
            / pl.col("QUANTITY")
        )
        .round(2)
        .alias("shelf_or_non_discounted_unit_ref"),
        ((pl.col("SALES_VALUE") - pl.col("COUPON_MATCH_DISC")) / pl.col("QUANTITY"))
        .round(2)
        .alias("non_loyalty_price"),
        (
            (pl.col("SALES_VALUE") - (pl.col("RETAIL_DISC") + pl.col("COUPON_MATCH_DISC")))
            / pl.col("QUANTITY")
        )
        .round(2)
        .alias("loyalty_card_price"),
    )
    .select(
        "household_key",
        "BASKET_ID",
        "DAY",
        "PRODUCT_ID",
        "QUANTITY",
        "SALES_VALUE",
        "RETAIL_DISC",
        "COUPON_DISC",
        "COUPON_MATCH_DISC",
        "loyalty_card_price",
        "non_loyalty_price",
    )
    .head(10)
    .collect(engine="streaming")
)
print(price_examples_df)

shape: (10, 11)
┌────────────┬────────────┬─────┬────────────┬───┬────────────┬────────────┬───────────┬───────────┐
│ household_ ┆ BASKET_ID  ┆ DAY ┆ PRODUCT_ID ┆ … ┆ COUPON_DIS ┆ COUPON_MAT ┆ loyalty_c ┆ non_loyal │
│ key        ┆ ---        ┆ --- ┆ ---        ┆   ┆ C          ┆ CH_DISC    ┆ ard_price ┆ ty_price  │
│ ---        ┆ i64        ┆ i64 ┆ i64        ┆   ┆ ---        ┆ ---        ┆ ---       ┆ ---       │
│ i64        ┆            ┆     ┆            ┆   ┆ f64        ┆ f64        ┆ f64       ┆ f64       │
╞════════════╪════════════╪═════╪════════════╪═══╪════════════╪════════════╪═══════════╪═══════════╡
│ 718        ┆ 2698536057 ┆ 1   ┆ 855325     ┆ … ┆ -0.4       ┆ -0.4       ┆ 2.19      ┆ 1.5       │
│            ┆ 1          ┆     ┆            ┆   ┆            ┆            ┆           ┆           │
│ 718        ┆ 2698536057 ┆ 1   ┆ 948756     ┆ … ┆ -0.75      ┆ -0.25      ┆ 1.09      ┆ 0.79      │
│            ┆ 1          ┆     ┆            ┆   ┆            ┆            

## 5. Estudo de Caso Oficial: Cliente `household_key = 208` (John Smith — PDF págs. 2 e 7–9)

Reconstrução das 8 campanhas recebidas por John Smith (`208`), seu perfil demográfico e os resgates de cupons (incluindo o cupom `51800000050` na Campanha 22 no Dia 654, cesta `41531980403`).

In [6]:
hh208_demo = demo_lf.filter(pl.col("household_key") == 208).collect()
print("Demografia de John Smith (household_key = 208):")
print(hh208_demo)

ct_lf = pl.scan_csv(settings.csv_path("campaign_table.csv"))
cd_lf = pl.scan_csv(settings.csv_path("campaign_desc.csv"))
cr_lf = pl.scan_csv(
    settings.csv_path("coupon_redempt.csv"),
    schema_overrides={"COUPON_UPC": pl.Utf8},
)

hh208_campaigns = (
    ct_lf.filter(pl.col("household_key") == 208)
    .join(cd_lf, on=["CAMPAIGN", "DESCRIPTION"])
    .sort("START_DAY")
    .collect()
)
print("\nCampanhas recebidas por John Smith (household_key = 208):")
print(hh208_campaigns)

hh208_redemptions = (
    cr_lf.filter(pl.col("household_key") == 208)
    .join(cd_lf, on="CAMPAIGN", how="left")
    .sort("DAY")
    .collect()
)
print("\nResgates de cupons de John Smith (incluindo Dia 654 / Campanha 22):")
print(hh208_redemptions)

hh208_day654 = (
    tx_lf.filter((pl.col("household_key") == 208) & (pl.col("DAY") == 654))
    .sort("PRODUCT_ID")
    .collect(engine="streaming")
)
print("\nCesta de compras de John Smith no Dia 654 (basket_id = 41531980403):")
print(hh208_day654)

Demografia de John Smith (household_key = 208):
shape: (1, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ classifica ┆ classifica ┆ classifica ┆ HOMEOWNER ┆ classific ┆ classific ┆ KID_CATEG ┆ household │
│ tion_1     ┆ tion_2     ┆ tion_3     ┆ _DESC     ┆ ation_5   ┆ ation_4   ┆ ORY_DESC  ┆ _key      │
│ ---        ┆ ---        ┆ ---        ┆ ---       ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│ str        ┆ str        ┆ str        ┆ str       ┆ str       ┆ str       ┆ str       ┆ i64       │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ Age Group4 ┆ Y          ┆ Level5     ┆ Homeowner ┆ Group5    ┆ 2         ┆ None/Unkn ┆ 208       │
│            ┆            ┆            ┆           ┆           ┆           ┆ own       ┆           │
└────────────┴────────────┴────────────┴───────────┴───────────┴───────────┴───────────┴───────────┘

Campanhas recebidas por John


Cesta de compras de John Smith no Dia 654 (basket_id = 41531980403):
shape: (18, 12)
┌─────────────┬────────────┬─────┬────────────┬───┬────────────┬─────────┬────────────┬────────────┐
│ household_k ┆ BASKET_ID  ┆ DAY ┆ PRODUCT_ID ┆ … ┆ TRANS_TIME ┆ WEEK_NO ┆ COUPON_DIS ┆ COUPON_MAT │
│ ey          ┆ ---        ┆ --- ┆ ---        ┆   ┆ ---        ┆ ---     ┆ C          ┆ CH_DISC    │
│ ---         ┆ i64        ┆ i64 ┆ i64        ┆   ┆ i64        ┆ i64     ┆ ---        ┆ ---        │
│ i64         ┆            ┆     ┆            ┆   ┆            ┆         ┆ f64        ┆ f64        │
╞═════════════╪════════════╪═════╪════════════╪═══╪════════════╪═════════╪════════════╪════════════╡
│ 208         ┆ 4153198040 ┆ 654 ┆ 827261     ┆ … ┆ 1043       ┆ 94      ┆ 0.0        ┆ 0.0        │
│             ┆ 3          ┆     ┆            ┆   ┆            ┆         ┆            ┆            │
│ 208         ┆ 4153198040 ┆ 654 ┆ 846823     ┆ … ┆ 1043       ┆ 94      ┆ 0.0        ┆ 0.0        │
│    